In [1]:
%cd '/home/mojtaba.kananisarche/Zh'

/home/mojtaba.kananisarche/Zh


In [2]:
from typing import Optional, List, Dict
import pandas as pd
from src.utils.file_folder import get_file_paths
import pandas as pd
import os

def _get_train_val_test_file_paths(
    input_path: str,
    primary_column_name: str,
    secondary_column_name: Optional[str] = None,
) -> List[Dict]:

    file_paths = get_file_paths(folder_path=input_path, depth=2, EXT_LIST=[".csv"])
    data = pd.DataFrame()
    data["path"] = file_paths
    data = data[~data['path'].apply(lambda x:os.path.split(x)[1].startswith('video_info'))].reset_index(drop=True)
    data["fold"] = (
        data["path"]
        .apply(lambda x: os.path.basename(x).split("_")[0])
        .astype(int)
        .astype(str)
    )
    data["subject_id"] = (
        data["path"]
        .apply(lambda x: os.path.basename(x).split("_")[1])
        .astype(int)
        .astype(str)
    )
    data = data.sort_values(["fold", "subject_id"]).reset_index(drop=True)

    columns_to_fold_by = data[
        [primary_column_name]
        + ([secondary_column_name] if secondary_column_name else [])
    ]

    train_val_test_file_paths = []
    unique_main_column_values = sorted(
        columns_to_fold_by[primary_column_name].unique()
    )

    if primary_column_name and secondary_column_name:
        for main_column_value in unique_main_column_values:
            corresponding_secondary_value = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ][secondary_column_name].unique()
            assert (
                len(corresponding_secondary_value) == 1
            ), f"Multiple Unique Secondary Values Found For Main Column Value {main_column_value} - {corresponding_secondary_value}"
            corresponding_secondary_value = corresponding_secondary_value[0]

            fold_train_val_test_file_paths = {"info": {}, "paths": {}}

            test_split = columns_to_fold_by[
                columns_to_fold_by[primary_column_name] == main_column_value
            ]
            fold_train_val_test_file_paths["paths"]["test"] = data.loc[
                test_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["test"] = {
                f"{primary_column_name}": sorted(
                    test_split[primary_column_name].unique().tolist()
                )
            }

            val_split = columns_to_fold_by[
                (columns_to_fold_by[primary_column_name] != main_column_value)
                & (
                    columns_to_fold_by[secondary_column_name]
                    == corresponding_secondary_value
                )
            ]
            fold_train_val_test_file_paths["paths"]["val"] = data.loc[
                val_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["val"] = {
                f"{primary_column_name}": sorted(
                    val_split[primary_column_name].unique().tolist()
                )
            }

            train_split = columns_to_fold_by[
                columns_to_fold_by[secondary_column_name]
                != corresponding_secondary_value
            ]
            fold_train_val_test_file_paths["paths"]["train"] = data.loc[
                train_split.index.to_list()
            ]["path"].to_list()
            fold_train_val_test_file_paths["info"]["train"] = {
                f"{primary_column_name}": sorted(
                    train_split[primary_column_name].unique().tolist()
                )
            }
            train_val_test_file_paths.append(fold_train_val_test_file_paths)

    elif primary_column_name:
        for main_column_value_i in unique_main_column_values:
            for main_column_value_j in unique_main_column_values:
                if main_column_value_i != main_column_value_j:
                    fold_train_val_test_file_paths = {"info": {}, "paths": {}}

                    test_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_i
                    ]
                    fold_train_val_test_file_paths["paths"]["test"] = data.loc[
                        test_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["test"] = {
                        f"{primary_column_name}": sorted(
                            test_split[primary_column_name].unique().tolist()
                        )
                    }

                    val_split = columns_to_fold_by[
                        columns_to_fold_by[primary_column_name]
                        == main_column_value_j
                    ]
                    fold_train_val_test_file_paths["paths"]["val"] = data.loc[
                        val_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["val"] = {
                        f"{primary_column_name}": sorted(
                            val_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_split = columns_to_fold_by[
                        ~columns_to_fold_by[primary_column_name].isin(
                            [main_column_value_i, main_column_value_j]
                        )
                    ]
                    fold_train_val_test_file_paths["paths"]["train"] = data.loc[
                        train_split.index.to_list()
                    ]["path"].to_list()
                    fold_train_val_test_file_paths["info"]["train"] = {
                        f"{primary_column_name}": sorted(
                            train_split[primary_column_name].unique().tolist()
                        )
                    }

                    train_val_test_file_paths.append(fold_train_val_test_file_paths)

    return train_val_test_file_paths

In [3]:
input_path = "/work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed"
primary_column_name = 'fold'
secondary_column_name = None
train_val_test_file_paths = _get_train_val_test_file_paths(
    input_path = input_path, 
    primary_column_name = primary_column_name, 
    secondary_column_name = secondary_column_name, 
)

In [ ]:
import numpy as np
import keras
from src.utils.common import get_csv_length, pop_row
from src.preprocess.reshape import count_slices_per_video, reshape_data_rnn
from src.utils.load_data import load_data, _extract_headers

class MyDataGenerator(keras.utils.Sequence):
    """
    Custom data generator for Keras.
    """

    def __init__(self, 
                 paths                  : List[str],
                 groupby_column         : str = "file_name",
                 sortby_column          : str = "frame",
                 target_column          : str = "target",
                 columns_to_drop        : List[str] = [],
                 batch_size             : int = 32,
                 slice_size             : int = 3000,
                 step_size              : int = 3000,
                 discard_threshold_ratio: float = 0.6): 
        """
        Initialization function.  This is where you set up the basic parameters
        of your data generator.
        """
        self.paths                   = paths
        self.pointer                 = 0
        self.groupby_column          = groupby_column
        self.sortby_column           = sortby_column
        self.target_column           = target_column
        self.columns_to_drop         = columns_to_drop
        self.columns_to_drop_1       = self._get_columns_to_drop_1()
        self.feature_columns         = self._get_feature_columns()
        self.batch_size              = batch_size
        self.slice_size              = slice_size
        self.step_size               = step_size
        self.discard_threshold_ratio = discard_threshold_ratio
        self.num_slices              = []
        self.data                    = pd.DataFrame(columns = ('X', 'y', 'g'))
        self.stash                   = pd.DataFrame(columns = ('X', 'y', 'g'))

        self.on_epoch_end()

    def _get_columns_to_drop_1(self):
        return [
            column_name
            for column_name in self.columns_to_drop
            if column_name not in [self.groupby_column, 
                                   self.sortby_column, 
                                   self.target_column]
        ]
    
    def _get_feature_columns(self):
        return [
            column_name
            for column_name in _extract_headers(self.paths[0])
            if column_name not in self.columns_to_drop
        ]
    
    def __len__(self):
        """
        Returns the number of batches per epoch.  Keras uses this to know how many
        times to call `__getitem__` in one epoch.
        """
        self.pointer = 0
        if not self.num_slices:
            video_info = pd.read_csv(os.path.join(input_path, 'video_info_1.csv'))
            file_lengths = video_info[video_info['path'].isin(self.paths)]['total_frames'].to_list()
            num_slices = count_slices_per_video(file_lengths, 
                                                slice_size              = self.slice_size,
                                                step_size               = self.step_size,
                                                discard_threshold_ratio = self.discard_threshold_ratio)
            self.num_slices = num_slices

        assert self.num_slices != len(self.paths), f"num_slices not calculated properly for all paths. num_slices : {len(self.num_slices)}, paths : {len(self.paths)}"
        return np.ceil(sum(self.num_slices)/self.batch_size)
    
    def _preprocess_data(self, file_data):
        file_data = file_data.drop(self.columns_to_drop_1, axis=1)
        X, y, g = reshape_data_rnn(df = file_data, 
                                   feature_columns=self.feature_columns, 
                                   slice_size=self.slice_size, 
                                   step_size=self.step_size, 
                                   groupby_column=self.groupby_column, 
                                   sortby_column=self.sortby_column, 
                                   target_column=self.target_column, 
                                   discard_threshold_ratio=self.discard_threshold_ratio)
        return X, y, g
    
    def _update_generator_memory(self, X_file, y_file, g_file):
        for (X, y, g) in zip(X_file, y_file, g_file):
            record_index = len(self.data)

            self.data.loc[record_index]  = {'X':'-', 'y':y, 'g':g}
            self.stash.loc[record_index] = {'X':X  , 'y':y, 'g':g}
    
    def on_epoch_end(self):
        """
        Called at the end of each epoch.  Use this to re-shuffle the data if needed.
        """
        pass  # Replace with your code

    def __data_generation(self):
        """
        Generates data containing batch_size samples. This is where the actual data
        loading and preprocessing happens.
        """
        while len(self.stash) < self.batch_size:
            file_data = load_data(input_path = self.paths[self.pointer])
            X_file, y_file, g_file = self._preprocess_data(file_data = file_data)
            self._update_generator_memory(X_file, y_file, g_file)
            self.pointer += 1

    def _pop_from_stash(self):
        popped, self.stash = pop_row(df = self.stash, 
                                     idx = slice(0,self.batch_size), 
                                     reset_index=False)
        return popped

    def __getitem__(self, index):
        """
        Generates one batch of data.  The core of the data generator.
        """
        self.__data_generation()
        popped = self._pop_from_stash()        
        return popped
        return (np.stack(popped['X']), np.stack(popped['y']))

In [151]:
columns_to_drop = [
        "file_name",
        "file_name_1",
        "type",
        "frame",
        "fold",
        "subject_id",
    ]
dg = MyDataGenerator(train_val_test_file_paths[0]['paths']['test'][:200], columns_to_drop=columns_to_drop, batch_size=7)

In [152]:
dg.__len__()

np.float64(171.0)

In [153]:
dg.__getitem__(0)

2025-04-22 05:24:37,487 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_0_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18069, 135)
2025-04-22 05:24:38,662 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_5_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (17985, 135)


,X,y,g
0,"[[2.0, 0.804, 0.005, 0.232, 0.04, 0.599, 1.208...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
1,"[[2.0, 0.975, 0.315, 0.186, 0.484, 1.249, 0.43...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
2,"[[2.0, -0.137, 0.268, 0.413, 0.714, 1.186, -0....",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
3,"[[2.0, 0.703, 0.149, 0.192, 0.053, 1.823, 0.55...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
4,"[[2.0, 0.237, 0.167, -0.129, 0.26, 1.162, 0.26...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
5,"[[2.0, 0.536, 0.584, 0.227, 0.304, 0.851, 0.31...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
6,"[[6.5, 1.412, 0.829, -0.153, -0.436, 1.186, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [145]:
dg.stash

,X,y,g
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
10,"[[6.5, 1.078, 0.301, -0.296, 0.095, 1.409, 0.7...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
11,"[[6.5, 1.022, 0.204, -0.076, -0.152, 0.835, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [154]:
dg.data

,X,y,g
0,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
1,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
2,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
3,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
4,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
5,,2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
6,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,,6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [147]:
dg.__getitem__(1)

2025-04-22 05:22:39,437 - INFO - Loaded file: /work/long_lab/Mojtaba/data/UTA/1.3.2-augmented_embedded_preprocessed/brightness_alpha0.8_beta-10/1_01_10_30fps_brightness_alpha0.8_beta-10_preprocessed.csv, data shape: (18027, 135)


,X,y,g
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
10,"[[6.5, 1.078, 0.301, -0.296, 0.095, 1.409, 0.7...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
11,"[[6.5, 1.022, 0.204, -0.076, -0.152, 0.835, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
12,"[[8.5, 0.723, 0.484, -0.139, 0.282, 1.075, 0.4...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
13,"[[8.5, 1.365, 0.639, -0.165, 0.032, 0.733, 0.5...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv


In [148]:
dg.data

,X,y,g
0,"[[2.0, 0.804, 0.005, 0.232, 0.04, 0.599, 1.208...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
1,"[[2.0, 0.975, 0.315, 0.186, 0.484, 1.249, 0.43...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
2,"[[2.0, -0.137, 0.268, 0.413, 0.714, 1.186, -0....",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
3,"[[2.0, 0.703, 0.149, 0.192, 0.053, 1.823, 0.55...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
4,"[[2.0, 0.237, 0.167, -0.129, 0.26, 1.162, 0.26...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
5,"[[2.0, 0.536, 0.584, 0.227, 0.304, 0.851, 0.31...",2.0,1_01_0_30fps_brightness_alpha0.8_beta-10.csv
6,"[[6.5, 1.412, 0.829, -0.153, -0.436, 1.186, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [149]:
dg.stash

,X,y,g
14,"[[8.5, -0.284, -0.589, 0.133, -0.255, 2.441, -...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
15,"[[8.5, 1.099, -0.034, 0.45, 0.533, 1.466, 0.14...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
16,"[[8.5, 1.115, 0.46, 0.475, -0.145, 1.436, 0.11...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv
17,"[[8.5, 1.565, 0.08, 0.382, 0.291, 1.759, 0.405...",8.5,1_01_10_30fps_brightness_alpha0.8_beta-10.csv


In [33]:
for (X, y, g) in zip(a1, a2, a3):
    print(X, y, g)
    break

[[ 2.     0.804  0.005 ...  0.205 -1.507 -0.229]
 [ 2.     0.522 -0.369 ...  0.199 -1.104 -0.297]
 [ 2.     0.492 -0.019 ...  1.232 -0.433  0.135]
 ...
 [ 2.     0.619  0.678 ...  0.419 -1.322  0.369]
 [ 2.     0.562  0.629 ...  0.454 -1.341  0.35 ]
 [ 2.     0.792  0.627 ...  0.625 -1.295  0.273]] 2.0 1_01_0_30fps_brightness_alpha0.8_beta-10.csv


In [40]:
a2[0]

np.float64(2.0)

In [155]:
from keras.utils import Sequence
import inspect

print(inspect.getfile(Sequence))

/home/mojtaba.kananisarche/.conda/envs/.cvenv/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py


In [161]:
dg.stash.sample(len(dg.stash))

,X,y,g
8,"[[6.5, 1.693, 1.159, -0.187, -0.224, 0.834, 1....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
9,"[[6.5, 0.975, 0.407, 0.227, -0.552, 1.425, 0.6...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
10,"[[6.5, 1.078, 0.301, -0.296, 0.095, 1.409, 0.7...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
7,"[[6.5, 1.418, 0.618, 0.09, -0.28, 0.878, 1.212...",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv
11,"[[6.5, 1.022, 0.204, -0.076, -0.152, 0.835, 0....",6.5,1_01_5_30fps_brightness_alpha0.8_beta-10.csv


In [ ]:
data = pd.read_csv(os.path.join(input_path, 'video_info_1.csv'))

In [73]:
popped, remaining = pop_row(data, slice(0,5))

In [74]:
popped

,path,total_frames
0,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18156
1,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18362
2,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18519
3,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,19954
4,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,16820


In [75]:
remaining

,path,total_frames
5,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18222
6,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,15010
7,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18481
8,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18729
9,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18471
...,...,...
1570,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18206
1571,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,17645
1572,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,18081
1573,/work/long_lab/Mojtaba/data/UTA/1.3.2-augmente...,20620


In [90]:
# %cd '/home/mojtaba.kananisarche/'

# from Zh.src.utils.load_data import load_data
# import numpy as np
# import pandas as pd
# from typing import List, Optional, Tuple, Optional
# from Zh.src.utils.file_folder import create_folders
# import os
# import json
# import pandas as pd
# import dataframe_image as dfi
# from Zh.src.utils.visualize_results import confusion_matrix_results, table_results

In [91]:
# path = "/home/mojtaba.kananisarche/Zh/outputs/2025-04-20|09:08:09|RNN_FACENET_5CV"

In [92]:
# data = table_results(
# results_directory_path  = path,
# output_directory_path   = None,
# output_file_path_prefix = 'results',
# cmap                    = "Blues",
# reverse_cmap            = False,
# header_1_bg_color       = 'silver',
# header_1_text_color     = 'black',
# header_2_bg_color       = 'darkgrey',
# header_2_text_color     = 'black',
# maximum_tries           = 1,
# slice_or_video          = 'video', #slice, video
# video_voting_method     = 'class_average', #majority, class_average, average
# height                  = 2,
# width                   = 5,
# )

In [93]:
# data = confusion_matrix_results(
#     results_directory_path  = path,
#     unique_classes          = [2, 6.5, 8.5],
#     output_directory_path   = None,
#     output_file_name_prefix = "results",
#     slice_or_video          = 'slice',     #slice, video
#     video_voting_method     = 'average',   #majority, class_average, average
#     index                   = -1,          # positive for any index and negative for all
#     split                   = "all",       # train, val, test, all
#     cmap                    = "Greys",
#     maximum_tries           = 5,
#     corner_text_color       = "white",
#     corner_bg_color         = "IndianRed",
#     pred_true_text_color    = "black",
#     pred_true_bg_color      = "silver",
#     labels_bg_color         = "beige",
#     labels_text_color       = "black",
#     height                  = 5,
#     width                   = None,
# )